# IC Analysis Pipeline
Computes **Weighted IC** (weighted Pearson correlation) between each feature and `y_target`,
per `ts_index`, per horizon.

**Output per horizon:**
- `ic_ts[h]`    — DataFrame of shape `(n_ts_index, 86)`, values = weighted IC
- `ic_stats[h]` — DataFrame of shape `(4, 86)`, rows = `[ic_mean, ic_std, ic_ir, n_periods]`

In [1]:
import pandas as pd
import numpy as np
import gc


## Config

In [8]:
TRAIN_PATH = "/home/lingod/kaggle_projects/ts_forecasting/data/train.parquet"
DATA_PATH = "/home/lingod/kaggle_projects/ts_forecasting/data/"
HORIZONS   = [1, 3, 10, 25]


## Load Data (minimal columns)

In [3]:
# Load only what we need: features + meta columns
# This avoids loading sub_code, id, etc. which are not needed
df_full = pd.read_parquet(TRAIN_PATH)

feature_cols = [c for c in df_full.columns if c.startswith('feature_')]
keep_cols    = feature_cols + ['ts_index', 'horizon', 'y_target', 'weight']
df_full      = df_full[keep_cols]

print(f"Loaded: {df_full.shape}")
print(f"Feature columns: {len(feature_cols)}")
print(f"ts_index range: {df_full['ts_index'].min()} – {df_full['ts_index'].max()}")
del df_full
gc.collect()


Loaded: (5337414, 90)
Feature columns: 86
ts_index range: 1 – 3601


0

## Weighted IC Function

In [4]:
def weighted_pearson_ic(group, feature, target='y_target', weight='weight'):
    """
    Compute weighted Pearson correlation between `feature` and `target`
    for a single ts_index group.

    - Drops rows where `feature` is NaN (per-feature null handling)
    - Keeps weight=0 rows (they contribute zero but are included)
    - Returns NaN if variance is zero (degenerate group)
    """
    # Drop nulls for this specific feature only
    valid = group[[feature, target, weight]].dropna(subset=[feature])

    x = valid[feature].values
    y = valid[target].values
    w = valid[weight].values

    w_sum = w.sum()
    if w_sum == 0 or len(x) < 2:
        return np.nan

    w_norm = w / w_sum

    mx = np.sum(w_norm * x)
    my = np.sum(w_norm * y)

    cov = np.sum(w_norm * (x - mx) * (y - my))
    sx  = np.sqrt(np.sum(w_norm * (x - mx) ** 2))
    sy  = np.sqrt(np.sum(w_norm * (y - my) ** 2))

    if sx < 1e-10 or sy < 1e-10:
        return np.nan

    return cov / (sx * sy)


## Main IC Pipeline

In [5]:
ic_ts    = {}   # horizon -> DataFrame (ts_index × features)
ic_stats = {}   # horizon -> DataFrame (stats × features)

for h in HORIZONS:
    print(f"\n{'='*60}")
    print(f"  Horizon = {h}")
    print(f"{'='*60}")

    # --- Query this horizon only (saves memory) ---
    df_h = pd.read_parquet(TRAIN_PATH).query("horizon == @h")
    print(f"  Rows: {len(df_h):,}  |  ts_index values: {df_h['ts_index'].nunique():,}")

    # --- Compute weighted IC per feature ---
    # Result: dict of {feature: pd.Series(index=ts_index, values=IC)}
    ic_dict = {}

    for i, feat in enumerate(feature_cols):
        if (i + 1) % 20 == 0 or (i + 1) == len(feature_cols):
            print(f"    Processing feature {i+1}/{len(feature_cols)}: {feat}")

        ic_series = (
            df_h
            .groupby('ts_index')
            .apply(weighted_pearson_ic, feature=feat)
        )
        ic_dict[feat] = ic_series

    # --- DF1: IC time series (ts_index × features) ---
    ic_ts[h] = pd.DataFrame(ic_dict)   # index=ts_index, columns=features
    ic_ts[h].index.name = 'ts_index'

    # --- DF2: IC statistics (stats × features) ---
    ic_mean = ic_ts[h].mean()
    ic_std  = ic_ts[h].std()
    ic_ir   = ic_mean / ic_std.replace(0, np.nan)   # preserve sign, avoid div-by-zero
    # n_periods: number of ts_index where IC was successfully computed (not NaN)
    n_periods = ic_ts[h].notna().sum()

    ic_stats[h] = pd.DataFrame(
        [ic_mean, ic_std, ic_ir, n_periods],
        index=['ic_mean', 'ic_std', 'ic_ir', 'n_periods']
    )

    # --- Summary ---
    print(f"\n  ic_ts[{h}]    shape: {ic_ts[h].shape}")
    print(f"  ic_stats[{h}] shape: {ic_stats[h].shape}")
    print(f"\n  Top 10 features by |IC IR| (horizon={h}):")
    top10 = ic_stats[h].loc['ic_ir'].abs().sort_values(ascending=False).head(10)
    for feat, val in top10.items():
        ic_m = ic_stats[h].loc['ic_mean', feat]
        ic_i = ic_stats[h].loc['ic_ir',   feat]
        print(f"    {feat:<20s}  ic_mean={ic_m:+.4f}  ic_ir={ic_i:+.4f}")

print("\n  Done.")



  Horizon = 1
  Rows: 1,394,653  |  ts_index values: 3,601
    Processing feature 20/86: feature_t
    Processing feature 40/86: feature_an
    Processing feature 60/86: feature_bh
    Processing feature 80/86: feature_cb
    Processing feature 86/86: feature_ch

  ic_ts[1]    shape: (3601, 86)
  ic_stats[1] shape: (4, 86)

  Top 10 features by |IC IR| (horizon=1):
    feature_cd            ic_mean=+0.0329  ic_ir=+0.2855
    feature_ca            ic_mean=+0.0324  ic_ir=+0.2620
    feature_bz            ic_mean=+0.0235  ic_ir=+0.2492
    feature_cb            ic_mean=+0.0257  ic_ir=+0.2464
    feature_bn            ic_mean=-0.0261  ic_ir=-0.2258
    feature_cc            ic_mean=+0.0238  ic_ir=+0.2242
    feature_u             ic_mean=-0.0250  ic_ir=-0.2241
    feature_by            ic_mean=+0.0218  ic_ir=+0.2158
    feature_v             ic_mean=-0.0201  ic_ir=-0.2061
    feature_am            ic_mean=-0.0211  ic_ir=-0.1894

  Horizon = 3
  Rows: 1,385,816  |  ts_index values: 3,601
 

## Quick Sanity Check

In [ ]:
for h in HORIZONS:
    ts_df  = ic_ts[h]
    st_df  = ic_stats[h]
    nan_pct = ts_df.isna().mean().mean() * 100
    print(f"Horizon {h:>2}  |  ic_ts shape: {ts_df.shape}  "
          f"|  ic_stats shape: {st_df.shape}  "
          f"|  NaN% in ic_ts: {nan_pct:.2f}%")


## Example: View ic_ts and ic_stats for Horizon 25

In [6]:
print("ic_ts[25] — first 5 rows:")
display(ic_ts[25].head())

print("\nic_stats[25]:")
display(ic_stats[25].round(4))


ic_ts[25] — first 5 rows:


,feature_a,feature_b,feature_c,feature_d,feature_e,feature_f,feature_g,feature_h,feature_i,feature_j,...,feature_by,feature_bz,feature_ca,feature_cb,feature_cc,feature_cd,feature_ce,feature_cf,feature_cg,feature_ch
ts_index,,,,,,,,,,,,,,,,,,,,,
1,0.181851,-0.010646,0.071461,0.051010,0.077135,0.083697,0.071798,0.021727,0.095016,0.148285,...,-0.280823,0.259736,0.295349,0.233303,0.255165,0.339935,-0.084245,0.057572,-0.207881,0.225931
2,0.082226,0.004863,-0.006013,-0.060105,0.039039,-0.016217,-0.021998,0.021296,-0.003573,0.034612,...,0.105890,0.256413,0.379108,0.307066,0.316511,0.407797,0.010413,-0.007175,-0.026301,0.071178
3,0.246164,-0.217008,-0.049070,-0.043982,0.062373,0.192741,-0.123039,0.067028,0.114077,0.217962,...,-0.198936,0.411691,0.410732,0.367109,0.300193,0.450085,-0.045762,0.054709,-0.288700,0.316194
4,0.131901,0.021316,0.045794,0.050670,-0.030121,0.083157,0.011672,-0.024331,0.100483,0.049139,...,0.182734,0.235564,0.319717,0.327533,0.229649,0.377451,-0.023283,0.075386,-0.097944,0.132066
5,0.193932,0.004568,-0.040244,0.057747,0.003202,0.003937,0.037992,-0.003210,0.103104,0.108399,...,-0.061058,0.195008,0.257786,0.206907,0.163832,0.331325,-0.100192,0.066816,-0.178155,0.203832



ic_stats[25]:


,feature_a,feature_b,feature_c,feature_d,feature_e,feature_f,feature_g,feature_h,feature_i,feature_j,...,feature_by,feature_bz,feature_ca,feature_cb,feature_cc,feature_cd,feature_ce,feature_cf,feature_cg,feature_ch
ic_mean,0.0085,0.0001,-0.0005,0.0006,-0.0015,0.0005,-0.0000,0.0128,-0.0190,0.0009,...,0.0263,0.0381,0.0687,0.0442,0.0443,0.0646,-0.0035,-0.0072,-0.0079,0.0081
ic_std,0.0601,0.0632,0.0635,0.0612,0.0618,0.0598,0.0622,0.0724,0.0791,0.0709,...,0.0830,0.0985,0.1351,0.1076,0.1049,0.1197,0.0765,0.0697,0.0698,0.0616
ic_ir,0.1409,0.0022,-0.0075,0.0091,-0.0248,0.0077,-0.0005,0.1773,-0.2397,0.0131,...,0.3167,0.3872,0.5086,0.4106,0.4224,0.5397,-0.0458,-0.1027,-0.1128,0.1315
n_periods,3601.0000,3601.0000,3601.0000,3601.0000,3601.0000,3601.0000,3601.0000,3601.0000,3601.0000,3601.0000,...,3601.0000,3601.0000,3601.0000,3601.0000,3601.0000,3601.0000,3595.0000,3601.0000,3595.0000,3601.0000


## Store ic_ts and ic_stats for tomorrow

In [ ]:
# store ic_ts and ic_stats to disk for later analysis
for h in HORIZONS:
    ts_df = ic_ts[h]
    st_df = ic_stats[h]

    ts_df.to_parquet(DATA_PATH+f"ic_ts_h{h}.parquet")
    st_df.to_parquet(DATA_PATH+f"ic_stats_h{h}.parquet")
